In [ ]:
# ======================
# IMPORTS
# ======================
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing import image

print("TensorFlow:", tf.__version__)

# ======================
# CONFIG
# ======================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = os.getcwd()
TRAIN_DIR = os.path.join(BASE_DIR, "dataset", "train")
VAL_DIR = os.path.join(BASE_DIR, "dataset", "val")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Train path:", TRAIN_DIR)
print("Val path:", VAL_DIR)

# ======================
# DATA GENERATOR
# ======================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Classes:", train_gen.class_indices)

# ======================
# SAVE LABELS
# ======================
labels_path = os.path.join(MODEL_DIR, "tomato_labels.json")

class_labels = {v: k for k, v in train_gen.class_indices.items()}

with open(labels_path, "w") as f:
    json.dump(class_labels, f, indent=4)

# ======================
# MODEL
# ======================
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ======================
# CALLBACKS
# ======================
checkpoint_path = os.path.join(MODEL_DIR, "tomato_model.keras")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ======================
# TRAINING
# ======================
model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen,
    callbacks=callbacks_list
)

# ======================
# FINE-TUNING
# ======================
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen,
    callbacks=callbacks_list
)

print("Training complete!")

# ======================
# PREDICTION FUNCTION
# ======================
def predict_disease(img_path):
    model = tf.keras.models.load_model(checkpoint_path)

    with open(labels_path, "r") as f:
        labels = json.load(f)

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]
    class_idx = np.argmax(preds)
    confidence = float(preds[class_idx])

    return {
        "disease": labels[str(class_idx)],
        "confidence": round(confidence * 100, 2)
    }

# Example:
# print(predict_disease("test.jpg"))

TensorFlow: 2.21.0
Train path: F:\krishi nova project ml part\tomato\dataset\train
Val path: F:\krishi nova project ml part\tomato\dataset\val
Found 10000 images belonging to 10 classes.
Found 1000 images belonging to 10 classes.
Classes: {'Tomato___Bacterial_spot': 0, 'Tomato___Early_blight': 1, 'Tomato___Late_blight': 2, 'Tomato___Leaf_Mold': 3, 'Tomato___Septoria_leaf_spot': 4, 'Tomato___Spider_mites Two-spotted_spider_mite': 5, 'Tomato___Target_Spot': 6, 'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 7, 'Tomato___Tomato_mosaic_virus': 8, 'Tomato___healthy': 9}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,385,197 (16.73 MB)

 Trainable params: 333,066 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.0959 - loss: 2.7210
Epoch 1: val_accuracy improved from None to 0.05200, saving model to F:\krishi nova project ml part\tomato\model\tomato_model.keras

Epoch 1: finished saving model to F:\krishi nova project ml part\tomato\model\tomato_model.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 974s 3s/step - accuracy: 0.1022 - loss: 2.6216 - val_accuracy: 0.0520 - val_loss: 2.3030
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1108 - loss: 2.4189
Epoch 2: val_accuracy improved from 0.05200 to 0.10000, saving model to F:\krishi nova project ml part\tomato\model\tomato_model.keras

Epoch 2: finished saving model to F:\krishi nova project ml part\tomato\model\tomato_model.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 476s 2s/step - accuracy: 0.1128 - loss: 2.3773 - val_accuracy: 0.1000 - val_loss: 2.3025
Epoch 3/3
174/313 ━━━━━━━━━━━━━━━━━━━━ 1:17 559ms/step - accuracy: 0.1197 - loss: 2.3211